# ⚡ Groq Multi-File RAG Studio — Google Colab Notebook

This updated notebook allows students to upload and ask questions from **multiple files**.

Supported file types:

```text
PDF, TXT, MD, XML, DOCX, CSV, XLSX, XLS
```

Complete pipeline:

```text
Multiple File Upload → Data Ingestion → Text Chunking → Embeddings → FAISS Vector DB → Groq RAG Answer → Appealing Gradio UI
```

Designed for PGD Generative AI classroom demonstration.

## Step 1 — Install Required Packages

Run this first. It installs LangChain, Groq, FAISS, Hugging Face embeddings, PDF, Word, Excel/CSV, and Gradio support.

In [23]:
%pip install -qU \
  langchain \
  langchain-core \
  langchain-groq \
  langchain-text-splitters \
  langchain-huggingface \
  langchain-community \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  python-docx \
  pandas \
  openpyxl \
  xlrd \
  gradio

## Step 2 — Add Your Groq API Key

Recommended Colab method:

1. Open **Secrets** from the left sidebar.
2. Add a secret named `GROQ_API_KEY`.
3. Paste your Groq API key.
4. Enable notebook access.

If the secret is not found, the notebook will securely ask you to enter the key.

In [24]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_key = userdata.get("Groq_API")
except Exception:
    groq_key = None

if not groq_key:
    groq_key = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = groq_key
os.environ["GROQ_MODEL"] = "openai/gpt-oss-120b"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"

print("Groq key loaded:", "Yes" if os.environ.get("GROQ_API_KEY") else "No")
print("Groq model:", os.environ["GROQ_MODEL"])
print("Embedding model:", os.environ["EMBEDDING_MODEL"])

Enter your Groq API key: ··········
Groq key loaded: Yes
Groq model: openai/gpt-oss-120b
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3 — Import Libraries and Create Folders

Uploaded files are stored in `/content/data/raw`. The FAISS vector database is stored in `/content/faiss_index`.

In [25]:
from pathlib import Path
import os
import xml.etree.ElementTree as ET

import pandas as pd
from pypdf import PdfReader
from docx import Document as DocxDocument

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

DATA_DIR = Path("/content/data/raw")
VECTOR_DB_PATH = Path("/content/faiss_index")

DATA_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md", ".xml", ".docx", ".csv", ".xlsx", ".xls"}

print("Libraries imported successfully.")
print("Data folder:", DATA_DIR)
print("Supported files:", sorted(SUPPORTED_EXTENSIONS))

Libraries imported successfully.
Data folder: /content/data/raw
Supported files: ['.csv', '.docx', '.md', '.pdf', '.txt', '.xls', '.xlsx', '.xml']


## Step 4 — Create Sample Multi-Format Data

This creates sample TXT and CSV files so the notebook can run even if students do not upload files.

In [26]:
sample_text = """Generative AI Course Notes

Generative AI is a type of artificial intelligence that can create new content such as text, images, code, audio, summaries, and conversations.

LangChain is a framework used to build applications with Large Language Models. It helps connect prompts, models, tools, memory, documents, chains, and retrieval systems.

A RAG system means Retrieval-Augmented Generation. It retrieves relevant information from documents and then generates an answer using an LLM.

The main steps of a simple RAG pipeline are:
1. Data ingestion: load documents.
2. Data transformation: split documents into chunks.
3. Embeddings: convert text chunks into numerical vectors.
4. Vector database: store and search embeddings.
5. Retrieval: find relevant chunks for a user question.
6. LLM answer generation: use retrieved chunks to answer the question.

Groq provides fast inference for open-source language models. Gradio is used to create a simple web interface for AI applications.

Attention is a mechanism used in transformer models. It helps the model focus on the most relevant words or tokens when generating a response.
"""

(DATA_DIR / "sample_ai_notes.txt").write_text(sample_text, encoding="utf-8")

pd.DataFrame(
    {
        "Student": ["Ali", "Sara", "Ahmed", "Ayesha"],
        "Topic": ["RAG", "LangChain", "Groq", "Embeddings"],
        "Score": [85, 90, 78, 88],
        "Remarks": [
            "Understands retrieval augmented generation",
            "Good understanding of chains and prompts",
            "Needs more practice with API keys",
            "Strong understanding of vector embeddings",
        ],
    }
).to_csv(DATA_DIR / "sample_students.csv", index=False)

print("Sample files created:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Sample files created:
- Digital Transformation of Anti Quackery Drives by SHCC.pdf
- sample_ai_notes.txt
- Digital Transformation of Anti Quackery Drives at SHCC.docx
- sample_students.csv


## Step 5 — Optional: Upload Multiple Files in Colab

Use this cell to upload multiple files before building the vector database.

Supported: `.pdf`, `.txt`, `.md`, `.xml`, `.docx`, `.csv`, `.xlsx`, `.xls`.

In [27]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    file_path = DATA_DIR / filename
    file_path.write_bytes(content)
    print("Uploaded:", file_path)

print("\nCurrent files in data/raw:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Saving Digital Transformation of Anti Quackery Drives by SHCC.pdf to Digital Transformation of Anti Quackery Drives by SHCC (1).pdf
Uploaded: /content/data/raw/Digital Transformation of Anti Quackery Drives by SHCC (1).pdf

Current files in data/raw:
- Digital Transformation of Anti Quackery Drives by SHCC.pdf
- sample_ai_notes.txt
- Digital Transformation of Anti Quackery Drives at SHCC.docx
- sample_students.csv
- Digital Transformation of Anti Quackery Drives by SHCC (1).pdf


## Step 6 — Multi-File Data Ingestion Functions

These loaders convert supported file types into LangChain `Document` objects.

In [28]:
def read_txt_or_md(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path):
    reader = PdfReader(str(path))
    docs = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(Document(page_content=text, metadata={"source": path.name, "page": page_number, "type": "pdf"}))
    return docs


def read_docx(path: Path) -> str:
    document = DocxDocument(str(path))
    parts = []
    for paragraph in document.paragraphs:
        if paragraph.text.strip():
            parts.append(paragraph.text.strip())
    for table in document.tables:
        for row in table.rows:
            row_text = " | ".join(cell.text.strip() for cell in row.cells)
            if row_text.strip():
                parts.append(row_text)
    return "\n".join(parts)


def read_xml(path: Path) -> str:
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        text_parts = []
        for element in root.iter():
            if element.text and element.text.strip():
                text_parts.append(element.text.strip())
        return "\n".join(text_parts)
    except Exception:
        return path.read_text(encoding="utf-8", errors="ignore")


def dataframe_to_text(df: pd.DataFrame, title: str) -> str:
    preview = df.head(100).fillna("").to_string(index=False)
    return f"""Tabular File: {title}

Rows: {len(df)}
Columns: {len(df.columns)}
Column Names: {", ".join(map(str, df.columns))}

Data Preview:
{preview}
"""


def read_csv(path: Path):
    df = pd.read_csv(path)
    return [Document(page_content=dataframe_to_text(df, path.name), metadata={"source": path.name, "type": "csv", "rows": len(df), "columns": len(df.columns)})]


def read_excel(path: Path):
    docs = []
    sheets = pd.read_excel(path, sheet_name=None)
    for sheet_name, df in sheets.items():
        docs.append(Document(page_content=dataframe_to_text(df, f"{path.name} | Sheet: {sheet_name}"), metadata={"source": path.name, "sheet": sheet_name, "type": "excel", "rows": len(df), "columns": len(df.columns)}))
    return docs


def load_single_file(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    if suffix in {".txt", ".md"}:
        return [Document(page_content=read_txt_or_md(path), metadata={"source": path.name, "type": suffix.replace(".", "")})]
    if suffix == ".docx":
        return [Document(page_content=read_docx(path), metadata={"source": path.name, "type": "docx"})]
    if suffix == ".xml":
        return [Document(page_content=read_xml(path), metadata={"source": path.name, "type": "xml"})]
    if suffix == ".csv":
        return read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return read_excel(path)
    return []


def load_documents_from_directory(directory=DATA_DIR):
    documents = []
    for path in sorted(Path(directory).rglob("*")):
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS:
            try:
                documents.extend(load_single_file(path))
            except Exception as e:
                print(f"Could not load {path.name}: {e}")
    return documents


documents = load_documents_from_directory(DATA_DIR)
print("STEP 1: MULTI-FILE DATA INGESTION")
print("=" * 60)
print("Documents loaded:", len(documents))
for i, doc in enumerate(documents[:5], start=1):
    print(f"\nDocument {i}")
    print("-" * 60)
    print("Source:", doc.metadata.get("source"))
    print("Type:", doc.metadata.get("type"))
    print(doc.page_content[:600])

STEP 1: MULTI-FILE DATA INGESTION
Documents loaded: 5

Document 1
------------------------------------------------------------
Source: Digital Transformation of Anti Quackery Drives at SHCC.docx
Type: docx
Digital Transformation of Anti-Quackery Field Monitoring
ACROSS SINDH using KoboTOOL BOX
The Sindh Healthcare Commission (SHCC) was established under the SHCC Act 2013, as a regulatory body to improve the quality of healthcare services and curb quackery in all its forms across Sindh. The Commission regulates public and private healthcare establishments throughout the province.
The Anti-Quackery Directorate is the frontline regulatory unit responsible for identifying quackery, taking enforcement measures, and ensuring compliance with applicable healthcare regulations.
SHCC conducts Anti-Quacker

Document 2
------------------------------------------------------------
Source: Digital Transformation of Anti Quackery Drives by SHCC (1).pdf
Type: pdf
DIGITAL TRANSFORMATION OF ANTI-QUACKERY

## Step 7 — Data Transformation / Chunking

Split documents into smaller chunks so FAISS can retrieve relevant parts.

In [29]:
def split_documents(documents, chunk_size=900, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    return splitter.split_documents(documents)

chunks = split_documents(documents, chunk_size=900, chunk_overlap=150)

print("STEP 2: DATA TRANSFORMATION")
print("=" * 60)
print("Original document objects:", len(documents))
print("Chunks created:", len(chunks))
for i, chunk in enumerate(chunks[:5], start=1):
    print(f"\nChunk {i}")
    print("-" * 60)
    print("Source:", chunk.metadata.get("source"))
    print("Type:", chunk.metadata.get("type"))
    print(chunk.page_content[:500])

STEP 2: DATA TRANSFORMATION
Original document objects: 5
Chunks created: 12

Chunk 1
------------------------------------------------------------
Source: Digital Transformation of Anti Quackery Drives at SHCC.docx
Type: docx
Digital Transformation of Anti-Quackery Field Monitoring
ACROSS SINDH using KoboTOOL BOX
The Sindh Healthcare Commission (SHCC) was established under the SHCC Act 2013, as a regulatory body to improve the quality of healthcare services and curb quackery in all its forms across Sindh. The Commission regulates public and private healthcare establishments throughout the province.
The Anti-Quackery Directorate is the frontline regulatory unit responsible for identifying quackery, taking enforcement 

Chunk 2
------------------------------------------------------------
Source: Digital Transformation of Anti Quackery Drives at SHCC.docx
Type: docx
SHCC conducts Anti-Quackery drives across Sindh. Previously, field officers relied on manual, paper-based forms, making data 

## Step 8 — Embeddings and FAISS Vector Database

Convert text chunks into numerical vectors and store them in FAISS.

In [30]:
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if len(chunks) == 0:
    raise ValueError("No chunks found. Upload files or create sample data first.")

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(str(VECTOR_DB_PATH))

print("STEP 3: EMBEDDINGS + FAISS")
print("=" * 60)
print("Embedding model:", EMBEDDING_MODEL)
print("Chunks embedded:", len(chunks))
print("Vector database saved at:", VECTOR_DB_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

STEP 3: EMBEDDINGS + FAISS
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chunks embedded: 12
Vector database saved at: /content/faiss_index


## Step 9 — Test Vector Search

Confirm FAISS can retrieve relevant chunks before using Groq.

In [31]:
question = "What is RAG?"
results = vector_store.similarity_search(question, k=2)

print("STEP 4: VECTOR SEARCH")
print("=" * 60)
print("Question:", question)
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 60)
    print("Source:", doc.metadata.get("source"))
    print("Type:", doc.metadata.get("type"))
    print(doc.page_content[:700])

STEP 4: VECTOR SEARCH
Question: What is RAG?

Result 1
------------------------------------------------------------
Source: sample_ai_notes.txt
Type: txt
Generative AI Course Notes

Generative AI is a type of artificial intelligence that can create new content such as text, images, code, audio, summaries, and conversations.

LangChain is a framework used to build applications with Large Language Models. It helps connect prompts, models, tools, memory, documents, chains, and retrieval systems.

A RAG system means Retrieval-Augmented Generation. It retrieves relevant information from documents and then generates an answer using an LLM.

The main steps of a simple RAG pipeline are:
1. Data ingestion: load documents.
2. Data transformation: split documents into chunks.
3. Embeddings: convert text chunks into numerical vectors.
4. Vector database:

Result 2
------------------------------------------------------------
Source: sample_ai_notes.txt
Type: txt
Groq provides fast inference for ope

## Step 10 — Groq RAG Chain

Question → Retrieve relevant chunks → Send context to Groq → Generate final answer.

In [32]:
def get_llm(temperature=0.1):
    return ChatGroq(
        model=os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b"),
        temperature=temperature,
        api_key=os.environ.get("GROQ_API_KEY")
    )


def get_source_label(doc):
    source = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page")
    sheet = doc.metadata.get("sheet")
    if page:
        return f"{source}, page {page}"
    if sheet:
        return f"{source}, sheet {sheet}"
    return source


def answer_with_groq(question, k=4, temperature=0.1):
    retrieved_docs = vector_store.similarity_search(question, k=k)
    context = "\n\n".join(f"[Source: {get_source_label(doc)}]\n{doc.page_content}" for doc in retrieved_docs)

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional PGD Generative AI teaching assistant. Answer using only the given context. Use simple and clear classroom language. If the answer is not in the context, say: 'I do not know from the uploaded documents.'"),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}\n\nGive a clear answer. Mention useful details from the context.")
    ])

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    sources = []
    for doc in retrieved_docs:
        label = get_source_label(doc)
        if label not in sources:
            sources.append(label)
    return answer, sources, context

answer, sources, context = answer_with_groq("What is LangChain?", k=4)
print("Groq Answer")
print("=" * 60)
print(answer)
print("\nSources:")
for source in sources:
    print("-", source)

Groq Answer
LangChain is a framework for building applications that use Large Language Models (LLMs). It lets you connect together prompts, models, tools, memory, documents, chains, and retrieval systems so you can create more powerful and flexible AI‑driven applications.

Sources:
- sample_ai_notes.txt
- Digital Transformation of Anti Quackery Drives by SHCC (1).pdf, page 1
- Digital Transformation of Anti Quackery Drives by SHCC.pdf, page 1


## Step 11 — Ask Your Own Question

Change the question and run again.

In [33]:
my_question = "What are the steps of a RAG pipeline?"
answer, sources, context = answer_with_groq(my_question, k=4, temperature=0.1)
print("Question:", my_question)
print("\nAnswer:")
print(answer)
print("\nSources:")
for source in sources:
    print("-", source)

Question: What are the steps of a RAG pipeline?

Answer:
The RAG (Retrieval‑Augmented Generation) pipeline works in six main steps:

1. **Data ingestion** – Load the source documents into the system.  
2. **Data transformation** – Split the loaded documents into smaller, manageable chunks.  
3. **Embeddings** – Convert each text chunk into a numerical vector (an embedding).  
4. **Vector database** – Store those vectors in a vector store so they can be efficiently searched.  
5. **Retrieval** – When a user asks a question, search the vector database to find the most relevant chunks.  
6. **LLM answer generation** – Feed the retrieved chunks to a large language model, which uses them to generate the final answer.  

These steps together let a system retrieve useful information from documents and then generate a response with a language model.

Sources:
- sample_ai_notes.txt
- Digital Transformation of Anti Quackery Drives at SHCC.docx
- Digital Transformation of Anti Quackery Drives by 

## Step 12 — Groq Chat Playground

Normal Groq chatbot without RAG.

In [34]:
def simple_groq_chat(message, temperature=0.4):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful Generative AI assistant. Explain concepts simply for students."),
        ("human", "{message}")
    ])
    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"message": message})

print(simple_groq_chat("Explain LangChain in simple words.", temperature=0.4))

**LangChain in a Nutshell**

Imagine you have a **robot helper** that can talk, write, search the web, do math, or even control other programs.  
All those abilities live in separate “tools” (like a calculator, a search engine, a knowledge‑base, etc.).  
**LangChain** is a set of building blocks that makes it easy to **link those tools together** and let a language model (like ChatGPT) decide when and how to use each one.

---

## 1. The Problem LangChain Solves

A plain language model is great at generating text, but it can’t:

| What a plain LLM can do | What it can’t do (or does poorly) |
|--------------------------|------------------------------------|
| Answer questions with the knowledge it was trained on | Look up fresh, real‑time information |
| Write a story | Perform a calculation with perfect accuracy |
| Summarize a document | Access a private database or trigger an API |

To give the model these extra powers, you have to **chain** (connect) different operations together. D

## Step 13 — Helper Functions for Gradio UI

The UI allows users to upload multiple files directly inside the app.

In [35]:
def normalize_gradio_files(files):
    if not files:
        return []
    paths = []
    for file in files:
        if isinstance(file, str):
            paths.append(Path(file))
        elif isinstance(file, dict) and "name" in file:
            paths.append(Path(file["name"]))
        elif hasattr(file, "name"):
            paths.append(Path(file.name))
    return paths


def build_vector_store_from_paths(paths, chunk_size=900, chunk_overlap=150):
    global documents, chunks, vector_store
    loaded_docs = []
    for path in paths:
        if path.suffix.lower() in SUPPORTED_EXTENSIONS:
            loaded_docs.extend(load_single_file(path))
    if not loaded_docs:
        raise ValueError("No supported files were uploaded.")
    documents = loaded_docs
    chunks = split_documents(documents, chunk_size=int(chunk_size), chunk_overlap=int(chunk_overlap))
    if len(chunks) == 0:
        raise ValueError("No text chunks were created from uploaded files.")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(str(VECTOR_DB_PATH))
    return len(documents), len(chunks)


def build_vector_store_from_existing_data(chunk_size=900, chunk_overlap=150):
    global documents, chunks, vector_store
    documents = load_documents_from_directory(DATA_DIR)
    chunks = split_documents(documents, chunk_size=int(chunk_size), chunk_overlap=int(chunk_overlap))
    if len(chunks) == 0:
        raise ValueError("No chunks found from existing data.")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    vector_store = FAISS.from_documents(chunks, embeddings)
    vector_store.save_local(str(VECTOR_DB_PATH))
    return len(documents), len(chunks)


def file_summary_markdown(docs):
    if not docs:
        return "No documents loaded."
    unique_rows = []
    for doc in docs:
        row = (doc.metadata.get("source", "unknown"), doc.metadata.get("type", "unknown"))
        if row not in unique_rows:
            unique_rows.append(row)
    text = "### Loaded Files\n\n| File | Type |\n|---|---|\n"
    for source, file_type in unique_rows:
        text += f"| {source} | {file_type} |\n"
    return text

print("UI helper functions ready.")

UI helper functions ready.


## Step 14 — More Appealing Gradio UI

Includes: multi-file upload, knowledge-base builder, RAG chatbot, retrieved-context viewer, Groq playground, and pipeline guide.

In [36]:
CUSTOM_CSS = """
.gradio-container { max-width: 1200px !important; margin: auto !important; }
#hero {
    padding: 30px;
    border-radius: 28px;
    background: radial-gradient(circle at top left, rgba(34,197,94,.35), transparent 28%), radial-gradient(circle at top right, rgba(56,189,248,.32), transparent 26%), linear-gradient(135deg, #020617, #0f172a 48%, #111827);
    color: white;
    box-shadow: 0 22px 70px rgba(15,23,42,.35);
    margin-bottom: 18px;
    border: 1px solid rgba(148,163,184,.28);
}
#hero h1 { font-size: 44px; margin-bottom: 8px; background: linear-gradient(90deg, #22c55e, #38bdf8, #a78bfa); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
#hero p { color: #cbd5e1; font-size: 17px; }
.success-box { padding: 16px; border-radius: 16px; background: linear-gradient(135deg, rgba(34,197,94,.16), rgba(16,185,129,.08)); border: 1px solid rgba(34,197,94,.45); }
.warning-box { padding: 16px; border-radius: 16px; background: linear-gradient(135deg, rgba(245,158,11,.16), rgba(251,191,36,.08)); border: 1px solid rgba(245,158,11,.45); }
"""


def ui_build_knowledge_base(files, use_existing, chunk_size, chunk_overlap):
    try:
        file_paths = normalize_gradio_files(files)
        if file_paths:
            doc_count, chunk_count = build_vector_store_from_paths(file_paths, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
            source_mode = "Uploaded files"
        elif use_existing:
            doc_count, chunk_count = build_vector_store_from_existing_data(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
            source_mode = "Existing /content/data/raw files"
        else:
            return "<div class='warning-box'>Please upload files or enable existing sample data.</div>", "No files loaded yet."

        status = f"""<div class='success-box'>
        <h3>✅ Knowledge Base Created</h3>
        <p><b>Source:</b> {source_mode}</p>
        <p><b>Document objects:</b> {doc_count}</p>
        <p><b>Chunks embedded:</b> {chunk_count}</p>
        <p><b>Vector DB:</b> {VECTOR_DB_PATH}</p>
        </div>"""
        return status, file_summary_markdown(documents)
    except Exception as e:
        return f"<div class='warning-box'><h3>⚠️ Build Failed</h3><p>{str(e)}</p></div>", "No knowledge base created."


def ui_rag_answer(question, k, temperature, show_context):
    if not question or not question.strip():
        return "Please write a question.", ""
    try:
        answer, sources, retrieved_context = answer_with_groq(question=question, k=int(k), temperature=float(temperature))
        source_text = "\n".join(f"- {source}" for source in sources)
        final_answer = f"## Answer\n\n{answer}\n\n## Sources\n{source_text}"
        context_output = retrieved_context if show_context else "Context hidden. Enable 'Show Retrieved Context' to view it."
        return final_answer, context_output
    except Exception as e:
        return f"## Error\n\n{str(e)}\n\n### Checklist\n1. Add Groq API key.\n2. Build the knowledge base first.\n3. Upload supported files.", ""


def ui_playground(message, temperature):
    if not message or not message.strip():
        return "Please write a prompt."
    try:
        return simple_groq_chat(message, temperature=float(temperature))
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(css=CUSTOM_CSS, title="Groq Multi-File RAG Studio", theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div id="hero">
        <h1>⚡ Groq Multi-File RAG Studio</h1>
        <p>Ask questions from PDFs, Word documents, Excel files, CSV files, TXT, MD, and XML using Groq + LangChain + FAISS.</p>
        <p><b>Classroom Flow:</b> Upload files → Build vector database → Ask questions → Explain retrieved context.</p>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("📁 Upload & Build Knowledge Base"):
            with gr.Row():
                with gr.Column(scale=2):
                    upload_files = gr.File(label="Upload Multiple Files", file_count="multiple", file_types=[".pdf", ".txt", ".md", ".xml", ".docx", ".csv", ".xlsx", ".xls"])
                    use_existing = gr.Checkbox(label="Use existing sample files from /content/data/raw if no upload is selected", value=True)
                    with gr.Row():
                        chunk_size_slider = gr.Slider(300, 2000, value=900, step=50, label="Chunk Size")
                        chunk_overlap_slider = gr.Slider(0, 400, value=150, step=25, label="Chunk Overlap")
                    build_button = gr.Button("⚡ Build Knowledge Base", variant="primary")
                with gr.Column(scale=1):
                    build_status = gr.HTML("<div class='warning-box'><h3>Knowledge Base Not Built Yet</h3><p>Upload files or use sample data, then click build.</p></div>")
                    file_summary = gr.Markdown("No files loaded yet.")
            build_button.click(fn=ui_build_knowledge_base, inputs=[upload_files, use_existing, chunk_size_slider, chunk_overlap_slider], outputs=[build_status, file_summary])

        with gr.Tab("💬 Ask Your Documents"):
            gr.Markdown("Ask questions from the currently built knowledge base.")
            question_box = gr.Textbox(label="Your Question", placeholder="Example: Summarize the Excel file. What is RAG? What are the main points in the Word document?", lines=3)
            with gr.Row():
                k_slider = gr.Slider(1, 10, value=4, step=1, label="Retrieved Chunks")
                temp_slider = gr.Slider(0, 1, value=0.1, step=0.1, label="Temperature")
                context_checkbox = gr.Checkbox(label="Show Retrieved Context", value=False)
            ask_button = gr.Button("Ask Groq", variant="primary")
            with gr.Row():
                answer_output = gr.Markdown(label="Answer")
                context_output = gr.Textbox(label="Retrieved Context", lines=18)
            ask_button.click(fn=ui_rag_answer, inputs=[question_box, k_slider, temp_slider, context_checkbox], outputs=[answer_output, context_output])
            question_box.submit(fn=ui_rag_answer, inputs=[question_box, k_slider, temp_slider, context_checkbox], outputs=[answer_output, context_output])

        with gr.Tab("🧠 Groq Chat Playground"):
            gr.Markdown("This tab uses Groq directly without documents. Use it to teach normal LLM prompting.")
            playground_prompt = gr.Textbox(label="Prompt", placeholder="Explain prompt engineering in simple words.", lines=5)
            playground_temperature = gr.Slider(0, 1, value=0.4, step=0.1, label="Temperature")
            playground_button = gr.Button("Generate with Groq", variant="primary")
            playground_output = gr.Textbox(label="Groq Response", lines=14)
            playground_button.click(fn=ui_playground, inputs=[playground_prompt, playground_temperature], outputs=playground_output)

        with gr.Tab("🧩 Pipeline Explanation"):
            gr.Markdown("""
            ## RAG Pipeline Used in This App

            ### 1. Multi-File Data Ingestion
            The app reads PDF, TXT, MD, XML, DOCX, CSV, XLSX, and XLS files.

            ### 2. Data Transformation
            Long documents are split into smaller chunks.

            ### 3. Embeddings
            Each chunk is converted into a numerical vector using Hugging Face embeddings.

            ### 4. FAISS Vector Database
            FAISS stores vectors and retrieves the most similar chunks.

            ### 5. Retrieval
            The app retrieves relevant chunks based on the user question.

            ### 6. Groq LLM
            Groq generates the final answer using the retrieved context.

            ### 7. Gradio UI
            Students interact with the complete RAG system through a clean browser interface.

            ## Student Practice
            - Upload one PDF and one CSV together.
            - Build the knowledge base.
            - Ask: "Summarize all uploaded files."
            - Ask: "What are the key points from the Word document?"
            - Ask: "What columns are present in the Excel/CSV file?"
            - Turn on retrieved context and explain how RAG works.
            """)

demo.launch(share=True)

/tmp/ipykernel_6116/665365397.py:65: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Groq Multi-File RAG Studio", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f7783f970284671f75.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Student Task

1. Upload **at least two different file types**, for example PDF + CSV or DOCX + XLSX.
2. Build the knowledge base.
3. Ask three questions from the uploaded files.
4. Enable retrieved context and explain which chunk helped answer the question.
5. Change temperature and compare the answers.
6. Explain the difference between normal Groq Chat and RAG Chat.